## TODO — Azure AI Content Understanding observability

This notebook makes a direct call to the Azure AI Content Understanding resource. It is not an agent invocation.

- Configure resource-level diagnostic logging later:
  - Azure Portal → `trusted-knowledge-intake-capstone-resource` → Monitoring → Diagnostic settings.
  - Send **Request and Response Logs**, **Audit Logs**, and **AllMetrics** to a Log Analytics workspace.
- Run one new learning request after diagnostics are enabled; existing requests cannot be logged retroactively.
- Review future request latency, operation name, and response outcome in Log Analytics.
- Decide data-retention, cost, and sensitive-data handling before enabling request/response logging.

Learning run already observed:
- source SHA-256: `21bf735feda38da23fe4d4308d063dfe9cbb2f9b125c1a1b6eeb215c5961c513`
- Content Understanding operation ID: `57dd7c3b-c933-472d-8a90-cd7517f1c767`
- warning: `FigureUnderstandingSkipped` because no completion-model deployment/default was resolved.

## Next focus — orchestration

Design the smallest deterministic orchestration path that carries source evidence, source hash, Content Understanding operation ID, and structured warnings into the Intake Analyst handoff without granting any approval authority.

In [1]:
import sys
print(sys.executable)

d:\ai-engineering\FrontierWeekTrustedKnowledgeIntake\.venv\Scripts\python.exe


In [2]:
from hashlib import sha256
import json
from pathlib import Path
from time import perf_counter

from azure.ai.contentunderstanding import ContentUnderstandingClient, to_llm_input
from azure.identity import AzureCliCredential

In [4]:
CONTENT_UNDERSTANDING_ENDPOINT = (
    "https://trusted-knowledge-intake-capstone-resource"
    ".cognitiveservices.azure.com/"
)
ANALYZER_ID = "prebuilt-documentSearch"

REPO_ROOT = Path.cwd()
PDF_PATH = REPO_ROOT / ".." / "data" / "sources" / "agentic-agile-v-2605.20456v1.pdf"

print(f"Notebook working directory: {REPO_ROOT}")
print(f"PDF path: {PDF_PATH}")
print(f"PDF found: {PDF_PATH.is_file()}")

Notebook working directory: d:\ai-engineering\FrontierWeekTrustedKnowledgeIntake\src
PDF path: d:\ai-engineering\FrontierWeekTrustedKnowledgeIntake\src\..\data\sources\agentic-agile-v-2605.20456v1.pdf
PDF found: True


In [5]:
credential = AzureCliCredential()

client = ContentUnderstandingClient(
    endpoint=CONTENT_UNDERSTANDING_ENDPOINT,
    credential=credential,
)

source_bytes = PDF_PATH.read_bytes()
source_sha256 = sha256(source_bytes).hexdigest()

print(f"Source SHA-256: {source_sha256}")
print(f"Source size: {len(source_bytes):,} bytes")

Source SHA-256: 21bf735feda38da23fe4d4308d063dfe9cbb2f9b125c1a1b6eeb215c5961c513
Source size: 175,406 bytes


In [6]:
started_at = perf_counter()

poller = client.begin_analyze_binary(
    analyzer_id=ANALYZER_ID,
    binary_input=source_bytes,
    content_type="application/pdf",
)

operation_id = poller.operation_id
print(f"Content Understanding operation ID: {operation_id}")

Content Understanding operation ID: 57dd7c3b-c933-472d-8a90-cd7517f1c767


In [7]:
analysis = poller.result()
elapsed_seconds = round(perf_counter() - started_at, 3)

print(f"Completed in {elapsed_seconds} seconds")
print(f"Analyzer: {analysis.analyzer_id}")
print(f"API version: {analysis.api_version}")
print(f"Content items: {len(analysis.contents or [])}")

Completed in 66.011 seconds
Analyzer: prebuilt-documentSearch
API version: 2026-06-01-preview
Content items: 1


In [8]:
warnings = [
    {
        "code": getattr(warning, "code", None),
        "message": getattr(warning, "message", None),
        "target": getattr(warning, "target", None),
    }
    for warning in (analysis.warnings or [])
]

run_record = {
    "source_sha256": source_sha256,
    "operation_id": operation_id,
    "analyzer_id": analysis.analyzer_id,
    "api_version": analysis.api_version,
    "created_at": str(getattr(analysis, "created_at", None)),
    "content_items": len(analysis.contents or []),
    "elapsed_seconds": elapsed_seconds,
    "warnings": warnings,
}

print(json.dumps(run_record, indent=2))

{
  "source_sha256": "21bf735feda38da23fe4d4308d063dfe9cbb2f9b125c1a1b6eeb215c5961c513",
  "operation_id": "57dd7c3b-c933-472d-8a90-cd7517f1c767",
  "analyzer_id": "prebuilt-documentSearch",
  "api_version": "2026-06-01-preview",
  "created_at": "2026-09-17 17:14:01+00:00",
  "content_items": 1,
  "elapsed_seconds": 66.011,
  "warnings": [
    {
      "code": "FigureUnderstandingSkipped",
      "message": "{\"status_code\":400,\"error_code\":\"ResourceError\",\"message\":\"This analyzer needs a 'completion' model deployment for current request, but none was resolved. Either 'models.completion' is not set on the analyzer, or the deployment it references is not registered for this resource. Configure it via 'PATCH /contentunderstanding/defaults'.\"}",
      "target": "models.completion"
    }
  ]
}


In [9]:
markdown = to_llm_input(analysis)
print(markdown[:5000])

---
mimeType: application/pdf
metadata:
  contentType: application/pdf
  pageCount: '7'
pages: 1-7
warnings:
- code: FigureUnderstandingSkipped
  message: '{"status_code":400,"error_code":"ResourceError","message":"This analyzer needs a ''completion'' model deployment for current request, but none was resolved. Either ''models.completion'' is not set on the analyzer, or the deployment it references is not registered for this resource. Configure it via ''PATCH /contentunderstanding/defaults''."}'
  target: models.completion
---
<!-- InputPageNumber: 1 -->

# Agentic Agile-V: From Vibe Coding to Verified Engineering in Software and Hardware Development

Christopher Koch
Independent Researcher

Abstract-Agentic AI coding systems can inspect repositories,
plan implementation steps, edit files, call tools, run tests, and
submit pull requests. These capabilities make software and
hardware development faster in some settings, but current
evidence does not support the simple claim that autonom